# SMA Crossover on SPY

Hello-world strategy: buy SPY when its 50-day moving average is above its 200-day; flat otherwise. The point of this notebook is to exercise the full repo pipeline (data load → signal → backtest → metrics → plot), not to discover alpha. SMA crossovers are well known *not* to beat buy-and-hold on SPY after costs.

See [`README.md`](README.md) for the writeup and [`backtest.py`](backtest.py) for the reproducible CLI version.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from tradinglib.backtest import run_backtest
from tradinglib.loaders.equities.yfinance import load_daily

SYMBOL = "SPY"
FAST = 50
SLOW = 200
START, END = "2010-01-01", "2024-12-31"

## Load data

In [ ]:
bars = load_daily(SYMBOL, start=START, end=END)
prices = bars["close"]
bars.head()

## Build the signal and inspect

In [ ]:
fast_ma = prices.rolling(FAST).mean()
slow_ma = prices.rolling(SLOW).mean()
signal = (fast_ma > slow_ma).astype(float)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
prices.plot(ax=axes[0], label="SPY close", color="black", alpha=0.7)
fast_ma.plot(ax=axes[0], label=f"{FAST}-day SMA")
slow_ma.plot(ax=axes[0], label=f"{SLOW}-day SMA")
axes[0].set_ylabel("Price ($)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
signal.plot(ax=axes[1], drawstyle="steps-post", color="steelblue")
axes[1].set_ylabel("Position")
axes[1].set_ylim(-0.1, 1.1)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Run the backtest

In [ ]:
result = run_backtest(prices, signal, fee_bps=1.0, slippage_bps=0.5)
pd.Series(result.metrics)

## Compare against buy-and-hold

In [ ]:
buy_hold = (1.0 + prices.pct_change().fillna(0.0)).cumprod() * result.config["initial_capital"]

fig, ax = plt.subplots(figsize=(12, 5))
result.equity_curve.plot(ax=ax, label=f"SMA {FAST}/{SLOW} crossover")
buy_hold.plot(ax=ax, label="Buy & hold", alpha=0.6)
ax.set_title(f"{SYMBOL} — SMA({FAST}/{SLOW}) crossover vs buy & hold")
ax.set_ylabel("Equity ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()